In [1]:
import sys
sys.path.append("..")

In [2]:
from src.metrics import pixel_accuracy

In [3]:
import torch

target = torch.tensor([0,1,1,2])
prediction = torch.tensor([0,1,1,2])

accu = pixel_accuracy(target, prediction)
print(accu)
type(accu)

1.0


float

In [4]:
target = torch.randint(low=0,high=2,size=(16,256,256))
prediction = torch.randint(low=0,high=2,size=(16,256,256))
accu = pixel_accuracy(target,prediction)
print(accu)

0.4998016357421875


In [5]:
from src.metrics import class_iou

In [6]:
target = torch.tensor([0,0,1,2,0,1])
prediction = torch.tensor([0,1,1,2,0,0])

class_iou(target, prediction, 2)

1.0

In [7]:
from src.metrics import class_dice, mean_iou

In [8]:
class_dice(target, prediction, 0)

0.6666666865348816

In [9]:
mean_iou(target, prediction)

0.6111111144224802

In [10]:
from src.metrics import SegmentationMetrics

In [14]:
metrics = SegmentationMetrics()
metrics.update(target, prediction)

In [17]:
seg_metric = metrics.compute()
seg_metric

{'pixel_acc': 0.6666666666666666,
 'iou': [0.5, 0.3333333333333333, 1.0],
 'dice': [0.6666666666666666, 0.5, 1.0],
 'miou': 0.611111111111111,
 'mdice': 0.7222222222222222}

In [20]:
metrics.reset()
metrics.update(target[:3], prediction[:3])
metrics.update(target[3:], prediction[3:])

In [21]:
seg_metric = metrics.compute()
seg_metric

{'pixel_acc': 0.6666666666666666,
 'iou': [0.5, 0.3333333333333333, 1.0],
 'dice': [0.6666666666666666, 0.5, 1.0],
 'miou': 0.611111111111111,
 'mdice': 0.7222222222222222}

In [1]:
import sys
sys.path.append("..")

In [3]:
import torch

input = torch.randn([16,64,256,256])
print(input.shape)

torch.Size([16, 64, 256, 256])


In [3]:
from src.model import DoubleConv

conv = DoubleConv(3,64)

In [5]:
(conv.forward(input)).shape

torch.Size([16, 64, 256, 256])

In [6]:
sum(p.numel() for p in conv.parameters())

38976

In [4]:
from src.model import Down

down = Down(64,128)
print((down.forward(input)).shape)

torch.Size([16, 128, 128, 128])


In [4]:
import torch
from src.model import Up

In [7]:
x = torch.randn([16,256,64,64])
skip = torch.randn([16,128,128,128])

up = Up(256,128)
(up.forward(x,skip)).shape

torch.Size([16, 128, 128, 128])

In [2]:
from src.model import UNet
import torch

x = torch.randn([16,3,256,256])

unet = UNet()
(unet.forward(x)).shape

torch.Size([16, 3, 256, 256])

In [3]:
import torch

device = "cuda"
model = UNet().to(device)
x = torch.randn(16, 3, 256, 256).to(device)

torch.cuda.reset_peak_memory_stats()

model.eval()
with torch.no_grad():
    out = model(x)

print("output shape:", out.shape)
print("peak memory (GB):", torch.cuda.max_memory_allocated() / 1e9)

output shape: torch.Size([16, 3, 256, 256])
peak memory (GB): 2.135545856


In [4]:
model.train()
torch.cuda.reset_peak_memory_stats()

out = model(x)
loss = out.mean()   # dummy loss, just to trigger backward
loss.backward()

print("training peak memory (GB):", torch.cuda.max_memory_allocated() / 1e9)

training peak memory (GB): 6.173254144


In [5]:
print(sum(p.numel() for p in UNet().parameters()))

31037763


In [6]:
props = torch.cuda.get_device_properties(0)
print(props.total_memory / 1e9)

6.438780928


In [7]:
x = torch.randn(8, 3, 256, 256).to(device)
model.train()
torch.cuda.reset_peak_memory_stats()
out = model(x)
out.mean().backward()
print("batch 8 training peak (GB):", torch.cuda.max_memory_allocated() / 1e9)

batch 8 training peak (GB): 3.281291776


In [1]:
import torch
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()

# Test 1: one pixel, correct class is 0
logits = torch.tensor([2.0, 1.0, 0.0]).reshape(1, 3, 1, 1)
target = torch.tensor([0]).reshape(1, 1, 1)
print(loss_fn(logits, target))

# Test 2: same pixel, correct class is 2
target = torch.tensor([2]).reshape(1, 1, 1)
print(loss_fn(logits, target))

# Test 3: random data in real shapes
logits = torch.randn(8, 3, 256, 256)
target = torch.randint(0, 3, (8, 256, 256))
print(loss_fn(logits, target))

tensor(0.4076)
tensor(2.4076)
tensor(1.3912)
